# M6.A6+ — 매출 예측 고도화 (다일 선행 · 예측 구간 · 학습 윈도우)

> 배경: 38차 담당자 확정 — **AI 책임 범위 = 매출 예측(+고도화)**. 메뉴 분해·재료 리스트업은 범위 외.
> 하네스: M6.A4 확정 규칙 · 기준 모델: V1-t(`05_model_selection.ipynb`) · 작성: 2026-07-27

🔒 **공개 저장소 데이터 정책** — 매출 절대 금액은 커밋하지 않는다. 출력 제거 상태로 추적, 본문은 상대값만.

**목적** — 확정된 1일 선행 점 예측(V1-t)을 서비스 요구에 맞게 확장·강화한다:
① **다일 선행(D+1~D+3)** — 선행일별 성능 계단 실측 ② **예측 구간(P10/P90)** — 발주 안전범위·신뢰도 입력
③ **학습 윈도우** — ml_pipeline §2 "슬라이딩 윈도우 적용 여부" 미확정 해소 ④ 앙상블 시도.
전부 선택 fold 5개에서만 평가 — **test(2026-04) 봉인 유지.**

## 판정 요약 (TL;DR)

1. **다일 선행 계단 확정** — 모델 우위(vs 같은 조건 MA-7)가 **D+1 -10.2% → D+2 -3.0% → D+3 +0.4%**로
   계단식 소멸. D+1은 모델이 확실, D+2는 소폭 우위, **D+3은 MA-7과 동급** → 1~3일 제공 시
   선행일별 신뢰도 차등(구간 확대·배지)이 필수. 원인: 핵심 신호가 최근 lag인데 선행이 길수록 낡아짐.
2. **예측 구간(P10/P90) 채택** — LightGBM quantile(비율 타깃 공간)로 산출한 80% 구간의 실측 커버리지
   **평균 78%**(fold별 72~89%) — 캘리브레이션 양호. 발주 안전범위와 M6.A8 신뢰도 기준의 입력으로 채택.
   단 **P50의 점 예측 대체는 기각**(+1.9%) — 점 예측은 l2 유지.
3. **학습 윈도우: expanding 확정** — rolling 90/120/180일은 **+15~17% 열세**, 특히 재개장 fold에서 붕괴
   (과거 regime 데이터를 버리면 표본 부족이 더 아픔). ml_pipeline §2 미확정("슬라이딩 윈도우") 해소:
   **미적용(전체 이력 학습)**, 재검토 조건 = 데이터 2년+ 축적 또는 뚜렷한 drift.
4. **앙상블(l2+l1 평균) 기각** — 이득 -0.2%로 무시 수준, 모델 2개 유지 비용만 증가.
5. **정직한 한계** — 일 단위 점 예측은 sMAPE 49%대의 노이즈 지배 구간(주문 중앙값 8~25건 소규모 매장).
   모델링 추가 이득보다 **데이터 레버**가 큼: ① 재개장 후 데이터 축적(최대 레버) ② 세담터 일별 유동인구
   ③ POS 상세내역의 채널(홀/배달)·시간대 분리 ④ 반일(점심/저녁) 타깃 분해. 순서대로 Phase 8+ 후보.

> 재현 노트 — 본 노트북은 h-안전 피처를 자체 재구성한다(h=1도 05와 컬럼 순서가 달라 colsample 추출이
> 달라짐 → ±1% 재현 오차). V1-t 공식치는 05 노트북이 SSOT.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp
import lightgbm as lgb

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open].copy()
y, tx = ob.total_amount, ob.tx_count
folds = pp.make_monthly_folds(ob.index)
SEL, TEST = folds[:-1], folds[-1]
print("선택 fold:", [f["month"] for f in SEL], "| test(봉인 — 미사용):", TEST["month"])

BEST = dict(learning_rate=0.0257, num_leaves=9, min_child_samples=10, subsample=0.7093,
            colsample_bytree=0.6796, reg_alpha=0.001, reg_lambda=0.0)  # 05 V1-t 채택값
STATIC = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
          "is_post_renewal", "days_since_reopen"]  # 달력·예보 가정 — 선행일과 무관

def build_h(h):
    """h영업일 선행용 피처·기준선 — 타깃 유래 피처는 전부 shift(h) 이후 정보만 사용."""
    s = y.copy()
    X = ob[STATIC].copy()
    X["lag_sales_h"] = s.shift(h)
    X["lag_tx_h"] = tx.shift(h)
    r7h = s.shift(h).rolling(7).mean()
    X["roll7_h"] = r7h
    X["roll_atv_h"] = (s / tx).shift(h).rolling(7).mean()
    bydow = s.groupby(s.index.dayofweek)
    X["lag_dow"] = bydow.shift(1)                          # 같은 요일 직전 — h≤7이면 안전
    X["roll4dow"] = bydow.apply(lambda g: g.shift(1).rolling(4).mean()).droplevel(0)
    X = pd.concat([X, pd.get_dummies(ob.index.dayofweek, prefix="dow").set_index(ob.index)], axis=1)
    b = X.select_dtypes(bool).columns
    X[b] = X[b].astype(int)
    return X, r7h

def mae(a, p): return float(np.mean(np.abs(a - p)))
def smape(a, p): return float(np.mean(2 * np.abs(a - p) / (a + np.abs(p))) * 100)
won = lambda x: f"{x:,.0f}"

def run_h(h, objective="l2", alpha=None, window=None):
    """비율 타깃 V1-t 구조를 h-안전 피처로 학습·평가. (평균 MAE, 평균 sMAPE, fold MAE, 예측 dict)"""
    X, r7h = build_h(h)
    ms, ss, preds = [], [], {}
    for f in SEL:
        tr, va = f["train"], f["val"]
        if window:
            tr = tr[tr > tr.max() - pd.Timedelta(days=window)]
        ytr = np.log1p(y.loc[tr]) - np.log1p(r7h.loc[tr])
        yva = np.log1p(y.loc[va]) - np.log1p(r7h.loc[va])
        k = ytr.notna()
        kw = dict(objective=objective)
        if alpha is not None:
            kw["alpha"] = alpha
        m = lgb.LGBMRegressor(n_estimators=800, random_state=42, verbosity=-1, **kw, **BEST)
        m.fit(X.loc[tr][k], ytr[k], eval_set=[(X.loc[va], yva)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        p = np.expm1(pd.Series(m.predict(X.loc[va]), index=va) + np.log1p(r7h.loc[va]))
        preds[f["month"]] = p
        ms.append(mae(y.loc[va], p))
        ss.append(smape(y.loc[va], p))
    return np.mean(ms), np.mean(ss), ms, preds

In [ ]:
# §1 다일 선행 h=1/2/3 — 같은 조건의 MA-7(h) 기준선과 비교
rows, curve = [], {}
for h in (1, 2, 3):
    m_, s_, ms, _ = run_h(h)
    _, r7h = build_h(h)
    naive = np.mean([mae(y.loc[f["val"]], r7h.loc[f["val"]]) for f in SEL])
    rows.append((f"D+{h}", won(m_), f"{s_:.1f}%", won(naive), f"{m_ / naive - 1:+.1%}"))
    curve[h] = (m_, naive)
display(pd.DataFrame(rows, columns=["선행", "모델 MAE(원)", "모델 sMAPE", "MA-7(h) MAE(원)", "모델 vs MA-7"]))

fig, ax = plt.subplots(figsize=(7, 3.4), constrained_layout=True)
hs = list(curve)
ax.plot(hs, [curve[h][0] / 1e4 for h in hs], marker="o", lw=2, color=PAL["blue"], label="V1-t 구조(모델)")
ax.plot(hs, [curve[h][1] / 1e4 for h in hs], marker="o", lw=2, color=PAL["orange"], label="MA-7(h) 기준선")
ax.set_xticks(hs, [f"D+{h}" for h in hs])
ax.set_ylabel("선택 fold 평균 MAE (만원)")
ax.set_title("선행일별 성능 계단 — 모델 우위는 D+1에 집중")
ax.legend(frameon=False, fontsize=9)
plt.show()

### §1 관찰 — 다일 선행

- 모델 우위: **D+1 -10.2% → D+2 -3.0% → D+3 +0.4%** — 선행이 길수록 핵심 신호(직전 매출 lag·최근 수준)가
  낡아져 우위가 계단식으로 소멸. D+3은 사실상 MA-7과 동급.
- **서빙 제안**: D+1은 모델 예측, D+2·3도 모델 값을 제공하되 **선행일별 신뢰도 차등**(구간 확대 + 배지) —
  M6.A8 신뢰도 산식에 선행일 항 포함. 야간 배치는 매일 돌므로 D+2·3 예측은 다음 날 자동 갱신됨(가장 최신
  D+1 예측이 항상 존재)을 UI에 안내하는 것이 정직한 설계.

In [ ]:
# §2 예측 구간 — quantile P10/P90 (비율 타깃 공간, 단조 변환이라 분위수 보존)
_, _, _, p10 = run_h(1, objective="quantile", alpha=0.10)
_, _, _, p50 = run_h(1, objective="quantile", alpha=0.50)
_, _, _, p90 = run_h(1, objective="quantile", alpha=0.90)
_, _, _, pl2 = run_h(1)

rows, cov_all = [], []
for f in SEL:
    m_ = f["month"]; a = y.loc[f["val"]]
    lo, hi = np.minimum(p10[m_], p90[m_]), np.maximum(p10[m_], p90[m_])  # 교차 시 정렬
    cov = float(((a >= lo) & (a <= hi)).mean()); cov_all.append(cov)
    rows.append((m_, f"{cov:.0%}", f"{float((hi - lo).median() / a.median()):.2f}x"))
display(pd.DataFrame(rows, columns=["fold", "P10~P90 커버리지(목표 80%)", "구간 폭(실측 중앙값 대비)"]))
p50m = np.mean([mae(y.loc[f["val"]], p50[f["month"]]) for f in SEL])
l2m = np.mean([mae(y.loc[f["val"]], pl2[f["month"]]) for f in SEL])
print(f"평균 커버리지 {np.mean(cov_all):.0%} | P50 점 예측 MAE는 l2 대비 {p50m / l2m - 1:+.1%} → 점 예측은 l2 유지")

# 마지막 선택 fold(2026-03) 밴드 시각화
f = SEL[-1]; m_ = f["month"]; a = y.loc[f["val"]]
lo, hi = np.minimum(p10[m_], p90[m_]), np.maximum(p10[m_], p90[m_])
fig, ax = plt.subplots(figsize=(10.5, 3.6), constrained_layout=True)
ax.fill_between(a.index, lo / 1e4, hi / 1e4, color=PAL["blue"], alpha=0.22, label="P10~P90 구간")
ax.plot(a.index, pl2[m_] / 1e4, lw=1.6, color=PAL["blue"], label="점 예측(l2)")
ax.scatter(a.index, a / 1e4, s=22, color=PAL["orange"], zorder=3, label="실측")
ax.set_ylabel("일 매출 (만원)")
ax.set_title(f"{m_} fold — 예측 구간과 실측 (재개장 직후 구간)")
ax.legend(frameon=False, fontsize=9)
plt.show()

### §2 관찰 — 예측 구간

- 80% 구간의 실측 커버리지 **평균 78%**(72~89%) — 별도 보정 없이 목표에 근접, **채택**.
- 구간 폭이 실측 중앙값의 1.2~1.6배로 넓은 것 자체가 이 데이터의 불확실성을 정직하게 표현 —
  점 하나만 주는 UI보다 "이 범위면 안전" 정보가 발주 의사결정에 실질적.
- 활용: ① 발주 안전범위(보수적 발주 = P10~P50 사이 정책) ② M6.A8 신뢰도 배지(구간 폭 상대값이
  임계 초과 시 "신뢰도 낮음") ③ D+2·3은 구간을 그대로 넓혀 차등 표현(§1).

In [ ]:
# §3 학습 윈도우 — expanding vs rolling(90/120/180d) · §4 앙상블
rows = []
for w in (None, 180, 120, 90):
    m_, _, _, _ = run_h(1, window=w)
    rows.append(("expanding(전체)" if w is None else f"최근 {w}일", m_))
base_w = rows[0][1]
tbl = pd.DataFrame([(n, won(v), "기준" if v == base_w else f"{v / base_w - 1:+.1%}") for n, v in rows],
                   columns=["윈도우", "MAE(원)", "vs expanding"])
display(tbl)

_, _, _, pb = run_h(1, objective="l1")
ens = np.mean([mae(y.loc[f["val"]], 0.5 * pl2[f["month"]] + 0.5 * pb[f["month"]]) for f in SEL])
print(f"앙상블 0.5·l2+0.5·l1: l2 단독 대비 {ens / l2m - 1:+.1%} → 기각")

### §3·§4 관찰 — 윈도우·앙상블

- **expanding 확정**: rolling 90~180일은 +15~17% 열세, 특히 재개장 fold에서 붕괴 — regime이 바뀌어도
  과거 데이터를 버리는 것보다 regime 피처로 흡수하는 편이 압도적으로 낫다(표본 256일 체제의 결론).
  `ml_pipeline.md` §2의 "슬라이딩 윈도우 적용 여부·창 크기" 미확정 → **미적용(expanding)으로 해소**.
  재검토 조건: 데이터 2년+ 축적 또는 MA-7 대비 skill의 지속 하락(drift).
- 앙상블(l2+l1 평균)은 -0.2%로 이득이 무시 수준 — 운영 모델 2개 유지 비용 대비 무익해 기각.

## §5 판정·다음 단계

**고도화 판정** — 채택: ① 다일 선행 제공(선행일별 신뢰도 차등 전제) ② P10/P90 예측 구간 ③ expanding 윈도우
확정. 기각: P50 점 예측 대체·앙상블·rolling 윈도우. 점 예측의 남은 이득은 모델링보다 **데이터 레버**
(축적 > 일별 유동인구 > 채널·시간대 분리)에 있음을 명시.

- spec 반영(docs PR): `ml_pipeline.md` §2 윈도우 확정 · `model_spec.md` §3 다일 계단·예측 구간 추가
- M6.A7 XAI: TreeSHAP 대상은 D+1 모델(V1-t). 구간·선행일 차등은 M6.A8 신뢰도 산식으로 연결
- 검수 3건 변동 없음 (`AI/data/README.md`)